# Gpt-image-1.5 提示词指南

## 1. 简介

`gpt-image-1.5` 是我们最新的图像生成模型，专为达到生产质量的视觉效果和高度可控的创意工作流而设计。它在真实感、准确性和可编辑性方面都有重大提升，非常适合专业设计任务和迭代式内容创作。与上一代相比，它在真实感、准确性和可编辑性方面都有重大提升，并同时支持高质量渲染和低延迟用例。

主要能力包括：

- **高保真照片级真实感**，具备自然的光照、准确的材质和丰富的色彩表现
- **灵活的质量与延迟权衡**，在较低设置下生成更快，同时视觉质量仍超过上一代图像模型
- **稳健的面部和身份保持**，适用于编辑、角色一致性和多步骤工作流
- **可靠的文字渲染**，图像中的字体清晰锐利、布局一致、对比鲜明
- **复杂的结构化视觉内容**，包括信息图、示意图和多格画面构图
- **精确的风格控制和风格迁移**，只需极少的提示词，即可支持从品牌设计系统到美术风格的各种需求
- **强大的现实世界知识和推理能力**，能够准确描绘物体、环境和场景

本指南重点介绍提示词模式、最佳实践，以及来自真实生产用例的示例提示词。


## 2. 提示词基础

* **结构 + 目标：** 按一致的顺序编写提示词（背景/场景 → 主体 → 关键细节 → 约束），并写明预期用途（广告、UI 样稿、信息图），以设定“模式”和精致程度。对于复杂的请求，使用简短的带标签段落或换行，而不是一整段长文本。

* **具体性 + 质量提示：** 具体说明材质、形状、纹理和视觉媒介（照片、水彩、3D 渲染），并且只在需要时加入有针对性的“质量杠杆”（例如 *胶片颗粒*、*带纹理的笔触*、*微距细节*）。对于照片级真实感，相机/构图术语（镜头、光圈感、光照）通常比笼统的“8K/超精细”更能可靠地提升真实感。

* **延迟与保真度**：对于延迟敏感或高并发量的用例，先从设置 quality="low" 开始，评估它是否满足你的视觉要求。在许多情况下，它能以显著更快的生成速度提供足够的保真度。

* **构图：** 指定取景和视角（特写、广角、俯视）、透视/角度（平视、低角度）以及光照/氛围（柔和漫射光、黄金时刻、高对比度）来控制画面。如果布局很重要，请明确指出位置（例如“logo 位于右上角”“主体居中，左侧留白”）。

* **约束（要修改什么、要保留什么）：** 明确说明排除项和不变项（例如“无水印”“无多余文字”“无 logo/商标”“保留身份/几何结构/布局/品牌元素”）。进行编辑时，使用“只修改 X”+“其他一切保持不变”，并在每次迭代中重复保留清单，以减少偏移。

* **图像中的文字：** 将字面文字放在**引号**中或使用**全大写**，并将排版细节（字体风格、大小、颜色、位置）作为约束指定。对于难处理的词（品牌名称、不常见的拼写），逐个字母拼写出来，以提高字符准确性。

* **多图输入：** 通过**序号和描述**引用每张输入图像（“图像 1：产品照片……图像 2：风格参考……”），并描述它们如何相互作用（“将图像 2 的风格应用到图像 1”）。进行合成时，要明确说明哪些元素移到哪里（“把图像 1 中的鸟放到图像 2 中的大象身上”）。

* **迭代而非堆砌：** 从一个简洁的基础提示词开始，然后通过一次只改一处的小幅后续指令进行完善（“让光线更暖一些”“去掉多余的那棵树”“恢复原来的背景”）。使用“与之前相同的风格”或“该主体”等指代来利用上下文，但如果关键细节开始偏移，请重新明确说明。

## 3. 设置

运行一次即可。它会：
- 创建 API 客户端
- 在 images 文件夹中创建 `output_images/`。
- 添加一个用于保存 base64 图像的小辅助函数

将编辑时使用的所有参考图像放入 `input_images/`（或更新示例中的路径）。

In [2]:
import os
import base64
from openai import OpenAI

client = OpenAI()

os.makedirs("../../images/input_images", exist_ok=True)
os.makedirs("../../images/output_images", exist_ok=True)

def save_image(result, filename: str) -> None:
    """
    Saves the first returned image to the given filename inside the output_images folder.
    """
    image_base64 = result.data[0].b64_json
    out_path = os.path.join("../../images/output_images", filename)
    with open(out_path, "wb") as f:
        f.write(base64.b64decode(image_base64))

## 4. 用例——生成（文本 → 图像）

## 4.1 信息图
使用信息图向特定受众（学生、高管、客户或普通大众）解释结构化信息。示例包括讲解图、海报、带标注的示意图、时间线和“视觉百科”素材。对于布局密集或图中文字较多的情况，建议将输出生成质量设置为 "high"。

In [3]:
prompt = """
Create a detailed Infographic of the functioning and flow of an automatic coffee machine like a Jura. 
From bean basket, to grinding, to scale, water tank, boiler, etc. 
I'd like to understand technically and visually the flow.
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt
)

save_image(result, "infographic_coffee_machine.png")

![](../../images/infographic_coffee_machine.png)

## 4.2 图像内翻译

用于将现有设计（广告、UI 截图、包装、信息图）本地化为另一种语言，而无需从头重建布局。关键是除文字之外保留一切——保持排版风格、位置、间距和层级一致——同时逐字准确地翻译，不添加多余词语，除非必要不重新排版，也不对 logo、图标或图像做任何意外修改。

In [4]:
prompt = """
Translate the text in the infographic to Spanish. Do not change any other aspect of the image.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/output_images/infographic_coffee_machine.png", "rb"),
    ],
    prompt=prompt
)


save_image(result, "infographic_coffee_machine_sp.png")

输出图像：

![](../../images/infographic_coffee_machine_sp.png)

## 4.3 感觉“自然”的照片级真实图像

要获得可信的照片级真实感，请像在描述一张当下真实拍摄的照片那样给模型下提示。使用摄影语言（镜头、光照、取景），并明确要求真实的纹理（毛孔、皱纹、织物磨损、瑕疵）。避免使用暗示影棚精修或摆拍的词语。当细节很重要时，设置 quality="high"。

In [6]:
prompt = """
Create a photorealistic candid photograph of an elderly sailor standing on a small fishing boat. 
He has weathered skin with visible wrinkles, pores, and sun texture, and a few faded traditional sailor tattoos on his arms. 
He is calmly adjusting a net while his dog sits nearby on the deck. Shot like a 35mm film photograph, medium close-up at eye level, using a 50mm lens. 
Soft coastal daylight, shallow depth of field, subtle film grain, natural color balance. 
The image should feel honest and unposed, with real skin texture, worn materials, and everyday detail. No glamorization, no heavy retouching. 
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt,
    quality="high"
)

save_image(result, "photorealism.png")

输出图像：

![](../../images/photorealism.png)

## 4.4 世界知识

GPT-image-1.5 内置推理能力和强大的世界知识。例如，当要求生成一个设定在 1969 年 8 月纽约州贝瑟尔的场景时，它能推断出伍德斯托克音乐节，并生成准确、符合语境的图像，而无需明确告知这一事件。

In [7]:
prompt = """
Create a realistic outdoor crowd scene in Bethel, New York on August 16, 1969.
Photorealistic, period-accurate clothing, staging, and environment.
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt
)

save_image(result, "world_knowledge.png")

输出图像：

![](../../images/world_knowledge.png)

## 4.5 Logo 生成

出色的 logo 生成来自清晰的品牌约束和简洁性。描述品牌的个性和用途，然后要求一个简洁、原创的标志，具有鲜明的形状、平衡的留白，并能在不同尺寸下良好缩放。

你可以指定参数 "n" 来表示想要生成的变体数量。

In [8]:
prompt = """
Create an original, non-infringing logo for a company called Field & Flour, a local bakery. 
The logo should feel warm, simple, and timeless. Use clean, vector-like shapes, a strong silhouette, and balanced negative space. 
Favor simplicity over detail so it reads clearly at small and large sizes. Flat design, minimal strokes, no gradients unless essential. 
Plain background. Deliver a single centered logo with generous padding. No watermark.
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt,
    n=4     # Generate 4 versions of the logo
)

# Save all 4 images to separate files
for i, item in enumerate(result.data, start=1):
    image_base64 = item.b64_json
    image_bytes = base64.b64decode(image_base64)
    with open(f"../../images/output_images/logo_generation_{i}.png", "wb") as f:
        f.write(image_bytes)

输出图像：

| 选项 1 | 选项 2 | 选项 3 | 选项 4 |
|:--------:|:--------:|:--------:|:--------:|
| ![](../../images/logo_generation_1.png) | ![](../../images/logo_generation_2.png) | ![](../../images/logo_generation_3.png) | ![](../../images/logo_generation_4.png)|

## 4.6 故事转连环漫画

对于故事转漫画的生成，将叙事定义为一系列清晰的视觉节拍，每格一个。保持描述具体且聚焦于动作，以便模型将故事转化为易读、节奏得当的分格画面。

In [9]:
prompt = """
Create a short vertical comic-style reel with 4 equal-sized panels.
Panel 1: The owner leaves through the front door. The pet is framed in the window behind them, small against the glass, eyes wide, paws pressed high, the house suddenly quiet.
Panel 2: The door clicks shut. Silence breaks. The pet slowly turns toward the empty house, posture shifting, eyes sharp with possibility.
Panel 3: The house transformed. The pet sprawls across the couch like it owns the place, crumbs nearby, sunlight cutting across the room like a spotlight.
Panel 4: The door opens. The pet is seated perfectly by the entrance, alert and composed, as if nothing happened.
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt
)

save_image(result, "comic_reel.png")

输出图像：

![](../../images/comic-reel.png)

## 4.7 UI 样稿

描述产品时，把它当作已经存在的产品，UI 样稿的效果最好。聚焦于布局、层级、间距和真实的界面元素，避免使用概念艺术类的措辞，这样结果看起来才会像一个可用的、已上线的界面，而不是设计草图。

In [11]:
prompt = """
Create a realistic mobile app UI mockup for a local farmers market. 
Show today’s market with a simple header, a short list of vendors with small photos and categories, a small “Today’s specials” section, and basic information for location and hours. 
Design it to be practical, and easy to use. White background, subtle natural accent colors, clear typography, and minimal decoration. 
It should look like a real, well-designed, beautiful app for a small local market. 
Place the UI mockup in an iPhone frame.
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt
)

save_image(result, "ui_farmers_market.png")

输出图像：

![](../../images/ui_farmers_market.png)

## 5. 用例——编辑（文本 + 图像 → 图像）

## 5.1 风格迁移

当你希望保留参考图像的*视觉语言*（调色板、纹理、笔触、胶片颗粒等），同时改变主体或场景时，风格迁移非常有用。为获得最佳效果，请描述哪些必须保持一致（风格线索）、哪些必须改变（新内容），并加入背景、取景以及“不添加额外元素”等硬性约束，以防止偏移。

In [12]:
prompt = """
Use the same style from the input image and generate a man riding a motorcycle on a white background.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/pixels.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "motorcycle.png")

输入图像：

![](../../images/pixels.png)

输出图像：

![](../../images/motorcycle.png)

## 5.2 虚拟试衣


虚拟试衣非常适合对身份保持要求严格的电商预览。关键在于明确锁定人物（面部、体型、姿势、发型、表情），*只*允许修改服装，然后要求真实的穿着效果（垂坠感、褶皱、遮挡）以及一致的光照/阴影，让服装看起来是自然穿在身上的，而不是贴上去的。

In [13]:
prompt = """
Edit the image to dress the woman using the provided clothing images. Do not change her face, facial features, skin tone, body shape, pose, or identity in any way. Preserve her exact likeness, expression, hairstyle, and proportions. Replace only the clothing, fitting the garments naturally to her existing pose and body geometry with realistic fabric behavior. Match lighting, shadows, and color temperature to the original photo so the outfit integrates photorealistically, without looking pasted on. Do not change the background, camera angle, framing, or image quality, and do not add accessories, text, logos, or watermarks.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/woman_in_museum.png", "rb"),
        open("../../images/input_images/tank_top.png", "rb"),
        open("../../images/input_images/jacket.png", "rb"),
        open("../../images/input_images/tank_top.png", "rb"),
        open("../../images/input_images/boots.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "outfit.png")

输入图像：

| 全身照 | 单品 1 |
|:------------:|:--------------:|
| ![](../../images/woman_in_museum.png)  | ![](../../images/jacket.png) |
| 单品 2 | 单品 3 |
| ![](../../images/tank_top.png) | ![](../../images/boots.png) |

输出图像：

<img src="../../images/outfit.png" width="400"/> 

## 5.3 草图 → 图像（渲染）


草图转渲染的工作流非常适合将粗略的草图转化为照片级真实的概念图，同时保留原始意图。把提示词当作规格说明来写：保留布局和透视，然后通过指定合理的材质、光照和环境来*增加真实感*。加入“不要添加新元素/文字”，以避免创造性的重新诠释。

In [14]:
prompt = """
Turn this drawing into a photorealistic image.
Preserve the exact layout, proportions, and perspective.
Choose realistic materials and lighting consistent with the sketch intent.
Do not add new elements or text.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/drawings.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "realistic_valley.png")

输入图像：

![](../../images/drawings.png)

输出图像：

![](../../images/realistic_valley.png)

## 5.4 产品样图（透明背景 + 标签完整性）


产品抠图和样图准备常用于商品目录、电商平台和设计系统。成功与否取决于边缘质量（轮廓干净，无毛边/光晕）和标签完整性（文字保持清晰且不被改动）。如果你想要真实感但不想重新设计风格，只需要求轻度润色，并可选择添加一个尊重 alpha 通道的细微接触阴影。

In [15]:
prompt = """
Extract the product from the input image.
Output: transparent background (RGBA PNG), crisp silhouette, no halos/fringing.
Preserve product geometry and label legibility exactly.
Optional: subtle, realistic contact shadow in the alpha (no hard cut line).
Do not restyle the product; only remove background and lightly polish.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/shampoo.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "extract_product.png")

输入图像：

![](../../images/shampoo.png)

输出图像：

![](../../images/extract_product.png)

## 5.5 带有真实图内文字的营销创意


带有真实图内文字的营销创意非常适合快速构思广告概念，但排版需要明确的约束。将确切的文案放在引号中，要求逐字渲染（不得有多余字符），并描述位置和字体风格。如果文字还原不够完美，请保持提示词严格并不断迭代——对措辞/布局做小幅调整通常能提升可读性。

In [16]:
prompt = """
Create a realistic billboard mockup of the shampoo on a highway scene during sunset.
Billboard text (EXACT, verbatim, no extra characters):
"Fresh and clean"
Typography: bold sans-serif, high contrast, centered, clean kerning.
Ensure text appears once and is perfectly legible.
No watermarks, no logos.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/shampoo.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "billboard.png")

输入图像：

![](../../images/shampoo.png)

输出图像：

![](../../images/billboard.png)

## 5.6 光照与天气转换


用于将一张照片重新布景为不同的氛围、季节或时段版本（例如晴天 → 阴天、白天 → 黄昏、晴朗 → 下雪），同时保持场景构图不变。关键是只改变环境条件——光照方向/质量、阴影、大气、降水和地面湿度——同时保留身份、几何结构、相机角度和物体位置，让它看起来仍是同一张原始照片。

In [17]:
prompt = """
Make it look like a winter evening with snowfall.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    input_fidelity="high", 
    quality="high",
    image=[
        open("../../images/output_images/billboard.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "billboard_winter.png")


输出图像：

![](../../images/billboard_winter.png)

## 5.7 物体移除


人物场景合成适用于分镜脚本、营销活动以及需要保持面部/身份的“假如”场景。通过指定写实的摄影风格（自然光照、可信的细节、无电影感调色）来锚定真实感，并锁定主体中不得改变的部分。如果可用，更高的输入保真度有助于在较大的场景编辑中保持人物相似度。

In [18]:
prompt = """
Remove the tree logo from the white t-shirt of the man. Do not change anything else.
"""

prompt = """
Remove the red stripes from the white t-shirt of the man. Do not change anything else.
"""

prompt = """
Change the color of thered hat to light blue as velvet. Do not change anything else.
"""


result = client.images.edit(
    model="gpt-image-1.5",
    input_fidelity="high", 
    quality="high",
    image=[
        open("../../images/output_images/man_with_blue_hat.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "man_with_no_flower.png")

| 原始输入 | 移除红色条纹 | 更换帽子颜色 |
|:------------:|:--------------:|:--------------:|
| ![](../../images/man_with_flower.png)  | ![](../../images/man_with_flower_no_stripes.png) |  ![](../../images/man_with_blue_hat.png) |

## 5.8 将人物插入场景


人物场景合成适用于分镜脚本、营销活动以及需要保持面部/身份的“假如”场景。通过指定写实的摄影风格（自然光照、可信的细节、无电影感调色）来锚定真实感，并锁定主体中不得改变的部分。如果可用，更高的输入保真度有助于在较大的场景编辑中保持人物相似度。

In [3]:
prompt = """
Generate a highly realistic action scene where this person is running away from a large, realistic brown bear attacking a campsite. The image should look like a real photograph someone could have taken, not an overly enhanced or cinematic movie-poster image.
She is centered in the image but looking away from the camera, wearing outdoorsy camping attire, with dirt on her face and tears in her clothing. She is clearly afraid but focused on escaping, running away from the bear as it destroys the campsite behind her.
The campsite is in Yosemite National Park, with believable natural details. The time of day is dusk, with natural lighting and realistic colors. Everything should feel grounded, authentic, and unstyled, as if captured in a real moment. Avoid cinematic lighting, dramatic color grading, or stylized composition.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    input_fidelity="high", 
    quality="high",
    image=[
        open("../../images/input_images/woman_in_museum.png", "rb"),
    ],
    prompt=prompt
)

save_image(result, "scene.png")

输出图像：

![](../../images/scene.png)

## 5.9 多图引用与合成


用于将多个输入中的元素组合成一张可信的图像——非常适合“把这个物体/人物放进那个场景”的工作流，而无需重新生成所有内容。关键是清楚地指定要移植什么（图像 2 中的狗）、放在哪里（紧挨着图像 1 中的女士）、哪些必须保持不变（场景、背景、取景），同时匹配光照、透视、比例和阴影，让合成结果看起来像是在原始照片中自然拍摄到的。

In [4]:
prompt = """
Place the dog from the second image into the setting of image 1, right next to the woman, use the same style of lighting, composition and background. Do not change anything else.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    input_fidelity="high", 
    quality="high",
    image=[
        open("../../images/output_images/test_woman.png", "rb"),
        open("../../images/output_images/test_woman_2.png", "rb"),
    ],
    prompt=prompt
)


save_image(result, "test_woman_with_dog.png")

| 输入图像 1 | 输入图像 2 | 输出 |
|:------------:|:--------------:|:--------------:|
| ![](../../images/test_woman.png)  | ![](../../images/test_woman_2.png) |  ![](../../images/test_woman_with_dog.png) |

## 6. 其他高价值用例

## 6.1 室内设计“替换”（精确编辑）
用于在真实空间中可视化家具或装饰的变化，而无需重新渲染整个场景。目标是精准的真实感：替换单个物体，同时保留相机角度、光照、阴影和周围环境，让编辑结果看起来像一张真实的照片，而不是一次重新设计。

In [5]:
prompt = """
In this room photo, replace ONLY white with chairs made of wood.
Preserve camera angle, room lighting, floor shadows, and surrounding objects.
Keep all other aspects of the image unchanged.
Photorealistic contact shadows and fabric texture.
"""

result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/input_images/kitchen.jpeg", "rb"),
    ],
    prompt=prompt
)

save_image(result, "kitchen-chairs.png")

| 输入图像 | 输出图像 |
|------------|--------------|
| ![](../../images/kitchen.jpeg) | ![](../../images/kitchen-chairs.png) |

## 6.2 3D 立体节日贺卡（产品风格样图）
非常适合季节性营销概念和印刷预览。强调触感上的真实感——纸张层次、纤维、折痕和柔和的影棚光——让结果看起来像一件被拍摄的实体产品，而不是一幅平面插画。

In [6]:
scene_description = (
    "a cozy Christmas scene with an old teddy bear sitting inside a keepsake box, "
    "slightly worn fur, soft stitching repairs, placed near a window with falling snow outside. "
    "The scene suggests the child has grown up, but the memories remain."
)

short_copy = "Merry Christmas — some memories never fade."

prompt = f"""
Create a Christmas holiday card illustration.

Scene:
{scene_description}

Mood:
Warm, nostalgic, gentle, emotional.

Style:
Premium holiday card photography, soft cinematic lighting,
realistic textures, shallow depth of field,
tasteful bokeh lights, high print-quality composition.

Constraints:
- Original artwork only
- No trademarks
- No watermarks
- No logos

Include ONLY this card text (verbatim):
"{short_copy}"
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt,
)

save_image(result, "christmas_holiday_card_teddy.png")

输出图像：

![](../../images/christmas_holiday_card_teddy.png)

## 6.3 收藏级手办 / 毛绒钥匙扣（周边概念）

用于早期周边构思和提案视觉稿。聚焦于高端产品摄影的要素（材质、包装、印刷清晰度），同时保持设计原创且不侵权。非常适合快速测试多个角色或包装变体。


In [7]:
# ---- Inputs ----
character_description = (
    "a vintage-style toy propeller airplane with rounded wings, "
    "a front-mounted spinning propeller, slightly worn paint edges, "
    "classic childhood proportions, designed as a nostalgic holiday collectible"
)

short_copy = "Christmas Memories Edition"

# ---- Prompt ----
prompt = f"""
Create a collectible action figure of {character_description}, in blister packaging.

Concept:
A nostalgic holiday collectible inspired by the simple toy airplanes
children used to play with during winter holidays.
Evokes warmth, imagination, and childhood wonder.

Style:
Premium toy photography, realistic plastic and painted metal textures,
studio lighting, shallow depth of field,
sharp label printing, high-end retail presentation.

Constraints:
- Original design only
- No trademarks
- No watermarks
- No logos

Include ONLY this packaging text (verbatim):
"{short_copy}"
"""

result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt,
)

save_image(result, "christmas_collectible_toy_airplane.png")

输出图像：

![](../../images/christmas_collectible_toy_airplane.png)

## 6.4 保持角色一致性的儿童绘本插画（多图工作流）
专为不允许角色偏移的多页插画流水线而设计。一个可复用的“角色锚点”可确保跨场景、姿势和页面的视觉连续性，同时允许环境和叙事上的变化。


1️⃣ 角色锚点——确立可复用的主角

目标：锁定角色的外观、比例、服装和基调。

In [8]:
# ---- Inputs ----
prompt = """
Create a children’s book illustration introducing a main character.

Character:
A young, storybook-style hero inspired by a little forest outlaw,
wearing a simple green hooded tunic, soft brown boots, and a small belt pouch.
The character has a kind expression, gentle eyes, and a brave but warm demeanor.
Carries a small wooden bow used only for helping, never harming.

Theme:
The character protects and rescues small forest animals like squirrels, birds, and rabbits.

Style:
Children’s book illustration, hand-painted watercolor look,
soft outlines, warm earthy colors, whimsical and friendly.
Proportions suitable for picture books (slightly oversized head, expressive face).

Constraints:
- Original character (no copyrighted characters)
- No text
- No watermarks
- Plain forest background to clearly showcase the character
"""

# ---- Image generation ----
result = client.images.generate(
    model="gpt-image-1.5",
    prompt=prompt,
)

save_image(result, "childrens_book_illustration_1.png")

输出图像：

![](../../images/childrens_book_illustration_1.png)

2️⃣ 故事延续——复用角色，推进叙事

目标：同一角色，新的场景 + 动作。
角色外观必须保持不变。

In [9]:
# ---- Inputs ----
prompt = """
Continue the children’s book story using the same character.

Scene:
The same young forest hero is gently helping a frightened squirrel
out of a fallen tree after a winter storm.
The character kneels beside the squirrel, offering reassurance.

Character Consistency:
- Same green hooded tunic
- Same facial features, proportions, and color palette
- Same gentle, heroic personality

Style:
Children’s book watercolor illustration,
soft lighting, snowy forest environment,
warm and comforting mood.

Constraints:
- Do not redesign the character
- No text
- No watermarks
"""
# ---- Image generation ----
result = client.images.edit(
    model="gpt-image-1.5",
    image=[
        open("../../images/output_images/childrens_book_illustration_1.png", "rb"),  # ← use image from step 1
    ],
    prompt=prompt,
)

save_image(result, "childrens_book_illustration_2.png")


输出图像：

![](../../images/childrens_book_illustration_2.png)

## 结论

在本 notebook 中，我们演示了如何使用 gpt-image-1.5 构建高质量、可控的图像生成和编辑工作流，使其能够胜任真实的生产环境。本 cookbook 强调将提示词结构、明确的约束以及小幅迭代修改作为控制真实感、布局、文字准确性和身份保持的主要手段。我们涵盖了生成和编辑两类模式——从信息图、照片级真实感、UI 样稿和 logo，到翻译、风格迁移、虚拟试衣、合成和光照变化。在所有示例中，本 cookbook 都强调了清楚区分哪些应当改变、哪些必须保持不变的重要性，以及在每次迭代中重申这些不变项以防止偏移的重要性。我们还强调了质量和输入保真度设置如何根据用例，在延迟与视觉精度之间进行有意识的权衡。这些示例共同构成了一套在生产图像工作流中部署 gpt-image-1.5 的实用、可复用的操作手册。